In [1]:
# =============================================================================
# FILE: run_all.py
# PURPOSE: End-to-end pipeline — generate → preprocess → analyse → visualise
# USAGE:  python run_all.py
# REQUIREMENTS: pip install numpy pandas matplotlib seaborn scikit-learn faker pyarrow
# =============================================================================

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "faker", "pyarrow"]:
    install(pkg)

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for CI/CD
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from faker import Faker
from scipy import stats
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (classification_report, roc_auc_score, roc_curve)
from sklearn.pipeline        import Pipeline
import random, string, warnings
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")

print("=" * 65)
print(" PAYMENT GATEWAY — TRANSACTION SUCCESS RATE DIAGNOSIS")
print(" Africa | Payments | 3-Week Sprint")
print("=" * 65)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — DATA GENERATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 1] Generating synthetic transaction dataset...")

SEED = 42
np.random.seed(SEED); random.seed(SEED)
fake = Faker("en_NG"); fake.seed_instance(SEED)

N   = 2_400_000
SD  = datetime(2024, 1, 1)
ED  = datetime(2024, 3, 31, 23, 59, 59)
TSR = 0.76

SCHEMES  = {"Verve": .35, "Visa": .35, "Mastercard": .30}
ISSUERS  = {"First Bank":.18,"GTBank":.17,"Zenith Bank":.16,
            "Access Bank":.15,"UBA":.13,"Stanbic IBTC":.08,
            "Fidelity Bank":.07,"Others":.06}
MERCH    = {"Fashion & Apparel":{"mcc":"5621","w":.22},
            "Electronics":{"mcc":"5734","w":.18},
            "Grocery & Food":{"mcc":"5411","w":.20},
            "Digital Services":{"mcc":"7372","w":.15},
            "Travel & Transport":{"mcc":"4722","w":.12},
            "Health & Beauty":{"mcc":"5912","w":.08},
            "Home & Garden":{"mcc":"5261","w":.05}}
ERRORS   = {"51":{"label":"Insufficient Funds","weight":.31,"ret":True},
            "96":{"label":"System Timeout",    "weight":.25,"ret":True},
            "05":{"label":"Do Not Honour",     "weight":.23,"ret":False},
            "14":{"label":"Invalid Card",      "weight":.06,"ret":False},
            "54":{"label":"Expired Card",      "weight":.05,"ret":False},
            "61":{"label":"Exceeds Limit",     "weight":.03,"ret":False},
            "91":{"label":"Issuer Unavailable","weight":.03,"ret":True},
            "57":{"label":"Txn Not Permitted", "weight":.02,"ret":False},
            "41":{"label":"Lost Card",         "weight":.01,"ret":False},
            "43":{"label":"Stolen Card",       "weight":.005,"ret":False},
            "XX":{"label":"Other",             "weight":.005,"ret":False}}

def hour_weights():
    b = np.ones(24)
    b[0:6]*=.2; b[6:9]*=.6; b[12:14]*=1.4; b[18:22]*=1.6; b[22:24]*=.5
    return b/b.sum()

HW = hour_weights()

print("  Sampling timestamps...")
day_off  = np.random.randint(0, 90, N)
hr_off   = np.random.choice(24, N, p=HW)
min_off  = np.random.randint(0, 60, N)
sec_off  = np.random.randint(0, 60, N)
ts = pd.Series([SD + timedelta(days=int(d),hours=int(h),minutes=int(m),seconds=int(s))
                for d,h,m,s in zip(day_off,hr_off,min_off,sec_off)]).sort_values().reset_index(drop=True)
day_sprint = (ts - SD).dt.days.values
hours      = ts.dt.hour.values

print("  Assigning attributes...")
schemes  = np.random.choice(list(SCHEMES), N, p=list(SCHEMES.values()))
issuers  = np.random.choice(list(ISSUERS), N, p=list(ISSUERS.values()))
cats     = list(MERCH); weights = [MERCH[c]["w"] for c in cats]
merch    = np.random.choice(cats, N, p=weights)
mccs     = np.array([MERCH[c]["mcc"] for c in merch])
amounts  = np.clip(np.random.lognormal(np.log(8500),.9,N), 100, 5_000_000).round(2)

print("  Computing outcomes...")
ekeys = list(ERRORS); ew = np.array([ERRORS[k]["weight"] for k in ekeys])
ew /= ew.sum()
status   = np.full(N,"success",dtype=object)
ecode    = np.full(N,"00",    dtype=object)
elabel   = np.full(N,"",     dtype=object)
is_retry = np.zeros(N,dtype=bool)
retry_ok = np.zeros(N,dtype=bool)

for i in range(N):
    p = TSR
    pf = 1-p
    if schemes[i]=="Verve" and merch[i]=="Grocery & Food" and day_sprint[i]>=45:
        pf = min(pf*3,.85)
    if issuers[i]=="GTBank" and 19<=hours[i]<=21:
        pf = min(pf*1.6,.90)
    if np.random.random()<pf:
        status[i] = "failed"
        ec = np.random.choice(ekeys, p=ew)
        ecode[i]  = ec; elabel[i] = ERRORS[ec]["label"]
        if ERRORS[ec]["ret"]:
            rp = {"51":.60,"96":.50,"91":.70}.get(ec,.30)
            sp = {"51":.40,"96":.30,"91":.50}.get(ec,.20)
            if np.random.random()<rp:
                is_retry[i]=True
                retry_ok[i]=(np.random.random()<sp)

is_failed  = (status=="failed").astype(int)
eff_status = np.where((status=="success")|retry_ok,"success","failed")
fsev = np.where(status=="success","SUCCESS",
       np.where(is_retry,"SOFT_FAILURE","HARD_FAILURE"))

df = pd.DataFrame({
    "timestamp":ts.values,"hour":hours,"day_of_sprint":day_sprint,
    "day_of_week":ts.dt.day_name().values,
    "merchant_category":merch,"mcc":mccs,"card_scheme":schemes,
    "issuing_bank":issuers,"amount_ngn":amounts,
    "status":status,"error_code":ecode,"error_label":elabel,
    "is_retry":is_retry,"retry_success":retry_ok,
    "is_failed":is_failed,"effective_status":eff_status,
    "failure_severity":fsev,
})
df["is_weekend"]  = ts.dt.dayofweek.values>=5
df["is_peak_hour"]= df["hour"].between(19,21)
df["scheme_code"] = df["card_scheme"].map({"Visa":0,"Mastercard":1,"Verve":2})
for b in ["First Bank","GTBank","Zenith Bank","Access Bank","UBA"]:
    df[f"bank_{b.replace(' ','_').lower()}"] = (df["issuing_bank"]==b).astype(int)
df["verve_grocery"]      = ((df["card_scheme"]=="Verve")&(df["merchant_category"]=="Grocery & Food")).astype(int)
df["post_day45"]         = (df["day_of_sprint"]>=45).astype(int)
df["verve_grocery_post45"]= df["verve_grocery"]*df["post_day45"]

n = len(df); fail = df["is_failed"].sum()
print(f"\n  Records: {n:,} | Raw TSR: {1-fail/n:.1%} | Effective TSR: {(df['effective_status']=='success').mean():.1%}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — PARETO TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 2] Computing Pareto table...")
failed_df = df[df["status"]=="failed"]
pareto = (
    failed_df.groupby(["error_code","error_label"])
    .agg(count=("amount_ngn","count"), tpv=("amount_ngn","sum"), ret=("is_retry","any"))
    .reset_index().sort_values("count",ascending=False).reset_index(drop=True)
)
pareto["pct"]     = pareto["count"]/len(failed_df)*100
pareto["cum_pct"] = pareto["pct"].cumsum()
pareto["monthly_tpv"] = pareto["tpv"]/3
print(pareto[["error_code","error_label","count","pct","cum_pct","ret"]].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — VISUALISATIONS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 3] Generating visualisations...")

PALETTE = {"success":"#2ECC71","soft":"#F39C12","hard":"#E74C3C",
           "primary":"#1A1A2E","accent":"#E94560","neutral":"#6C757D"}
sns.set_style("whitegrid")
plt.rcParams.update({"font.size":10,"axes.titlesize":13,
                      "axes.titleweight":"bold","figure.dpi":120})

# Fig 1 — Overview Dashboard
fig, axes = plt.subplots(2,2,figsize=(17,11))
fig.suptitle("Payment Gateway — Transaction Diagnostic Dashboard\n90 Days | 2.4M Transactions",
             fontsize=15,fontweight="bold",y=.98)

# A — Donut
ax = axes[0,0]
sc = df["failure_severity"].value_counts()
ax.pie([sc.get("SUCCESS",0),sc.get("SOFT_FAILURE",0),sc.get("HARD_FAILURE",0)],
       labels=["Approved","Soft Failure\n(Retriable)","Hard Failure\n(Permanent)"],
       colors=[PALETTE["success"],PALETTE["soft"],PALETTE["hard"]],
       autopct="%1.1f%%",startangle=90,
       wedgeprops=dict(width=0.6,edgecolor="white",linewidth=2),textprops={"fontsize":9})
ax.set_title("(A) Outcome Distribution")

# B — TSR by scheme
ax = axes[0,1]
st = df.groupby("card_scheme")["is_failed"].apply(lambda x:(1-x.mean())*100).reset_index()
sc_colors = {"Visa":"#1A1F71","Mastercard":"#EB001B","Verve":"#006B54"}
bars = ax.bar(st["card_scheme"],st["is_failed"],
              color=[sc_colors[s] for s in st["card_scheme"]],width=.5,edgecolor="white")
ax.axhline(90,color=PALETTE["accent"],ls="--",lw=1.5,label="Target 90%")
ax.axhline(76,color=PALETTE["neutral"],ls=":",lw=1.5,label="Baseline 76%")
for b in bars:
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+.3,f"{b.get_height():.1f}%",
            ha="center",fontweight="bold",fontsize=11)
ax.set_ylim(60,100); ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("(B) TSR by Card Scheme")

# C — Hourly volume & failure rate
ax = axes[1,0]; ax2 = ax.twinx()
hourly = df.groupby("hour").agg(vol=("is_failed","count"),fail=("is_failed","sum")).reset_index()
hourly["fr"] = hourly["fail"]/hourly["vol"]*100
ax.bar(hourly["hour"],hourly["vol"]/1000,color=PALETTE["primary"],alpha=.35,label="Volume (K)")
ax2.plot(hourly["hour"],hourly["fr"],color=PALETTE["accent"],lw=2.5,marker="o",ms=4,label="Failure Rate")
ax.axvspan(19,21,alpha=.1,color=PALETTE["accent"])
ax.text(20,hourly["vol"].max()/1000*.85,"Peak",ha="center",fontsize=8,color=PALETTE["accent"])
ax.set_xlabel("Hour of Day"); ax.set_ylabel("Volume (K)"); ax2.set_ylabel("Failure Rate (%)")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("(C) Hourly Volume vs. Failure Rate")
lines1,labs1 = ax.get_legend_handles_labels(); lines2,labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labs1+labs2,fontsize=9,loc="upper left")

# D — Amount distribution
ax = axes[1,1]
for sev,col in [("SUCCESS",PALETTE["success"]),("SOFT_FAILURE",PALETTE["soft"]),("HARD_FAILURE",PALETTE["hard"])]:
    v = df[df["failure_severity"]==sev]["amount_ngn"]
    ax.hist(np.log10(v+1),bins=50,alpha=.5,color=col,label=sev.replace("_"," ").title(),density=True)
ax.set_xlabel("Amount (log₁₀ NGN)"); ax.set_ylabel("Density")
xt=[2,3,4,5,6]; ax.set_xticks(xt); ax.set_xticklabels([f"₦{10**x:,.0f}" for x in xt],fontsize=8)
ax.legend(fontsize=9); ax.set_title("(D) Amount Distribution by Outcome")

plt.tight_layout(); fig.savefig("fig1_overview_dashboard.png",dpi=150,bbox_inches="tight")
print("  Saved: fig1_overview_dashboard.png")
plt.close()

# Fig 2 — Pareto Chart
fig,ax1 = plt.subplots(figsize=(14,7)); ax2 = ax1.twinx()
bc = [PALETTE["hard"] if i<3 else "#ADB5BD" for i in range(len(pareto))]
bars = ax1.bar(pareto["error_label"],pareto["pct"],color=bc,edgecolor="white",lw=1.2,zorder=3)
ax2.plot(pareto["error_label"],pareto["cum_pct"],color=PALETTE["accent"],lw=2.5,
         marker="D",ms=7,zorder=4,label="Cumulative %")
ax2.axhline(80,color=PALETTE["primary"],ls="--",lw=1.5,alpha=.7,label="80% threshold")
for b,row in zip(bars,pareto.itertuples()):
    ax1.text(b.get_x()+b.get_width()/2,b.get_height()+.4,f"{row.pct:.1f}%",
             ha="center",va="bottom",fontsize=9,fontweight="bold")
ax1.set_ylabel("% of Total Failures"); ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0,110); ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=28,ha="right",fontsize=9); ax2.legend(loc="center right",fontsize=10)
ax1.axvspan(-.5,2.5,alpha=.07,color=PALETTE["hard"])
ax1.text(1,pareto["pct"].max()*.6,"Vital Few\n(3 codes → 79%)",
         ha="center",fontsize=11,color=PALETTE["hard"],fontweight="bold")
plt.title("Pareto Analysis — Payment Failure Root Causes",fontsize=14,fontweight="bold",pad=15)
plt.tight_layout(); fig.savefig("fig2_pareto_chart.png",dpi=150,bbox_inches="tight")
print("  Saved: fig2_pareto_chart.png"); plt.close()

# Fig 3 — Issuer × Scheme Heatmap
pivot = df.groupby(["issuing_bank","card_scheme"])["is_failed"].mean().unstack("card_scheme").mul(100)
fig,ax = plt.subplots(figsize=(10,6))
sns.heatmap(pivot,annot=True,fmt=".1f",cmap="RdYlGn_r",linewidths=.5,linecolor="white",
            cbar_kws={"label":"Failure Rate (%)"},ax=ax,vmin=15,vmax=40)
ax.set_title("Failure Rate Heatmap: Issuing Bank × Card Scheme (%)",fontweight="bold")
plt.tight_layout(); fig.savefig("fig3_issuer_scheme_heatmap.png",dpi=150,bbox_inches="tight")
print("  Saved: fig3_issuer_scheme_heatmap.png"); plt.close()

# Fig 4 — MCC × Scheme Heatmap
pivot2 = df.groupby(["merchant_category","card_scheme"])["is_failed"].mean().unstack("card_scheme").mul(100)
fig,ax = plt.subplots(figsize=(11,6))
sns.heatmap(pivot2,annot=True,fmt=".1f",cmap="RdYlGn_r",linewidths=.5,linecolor="white",
            cbar_kws={"label":"Failure Rate (%)"},ax=ax,vmin=15,vmax=55)
ax.set_title("Failure Rate Heatmap: Merchant Category × Card Scheme\n(Verve × Grocery block clearly visible)",fontweight="bold")
plt.tight_layout(); fig.savefig("fig4_mcc_scheme_heatmap.png",dpi=150,bbox_inches="tight")
print("  Saved: fig4_mcc_scheme_heatmap.png"); plt.close()

# Fig 5 — Retry Segmentation
failed2 = df[df["status"]=="failed"]
rs = failed2.groupby("error_code").agg(
    total=("is_retry","count"), retried=("is_retry","sum"), succ=("retry_success","sum")
).reset_index()
rs2 = rs[rs["error_code"].isin(["51","96","91"])].copy()
not_ret = rs2["total"]-rs2["retried"]; ret_fail = rs2["retried"]-rs2["succ"]; ret_ok = rs2["succ"]
fig,axes = plt.subplots(1,2,figsize=(15,6))
fig.suptitle("Retry Segmentation — Soft vs. Hard Failure Decomposition",fontsize=14,fontweight="bold")
ax = axes[0]; x = range(len(rs2))
ax.bar(x,not_ret,label="Not Retried (Hard)",color=PALETTE["hard"],alpha=.85)
ax.bar(x,ret_fail,bottom=not_ret,label="Retried — Still Failed",color=PALETTE["soft"],alpha=.85)
ax.bar(x,ret_ok,bottom=not_ret+ret_fail,label="Retried — Succeeded",color=PALETTE["success"],alpha=.85)
ax.set_xticks(list(x)); ax.set_xticklabels([f"Error {c}" for c in rs2["error_code"]])
ax.set_ylabel("Transactions"); ax.legend(fontsize=9); ax.set_title("(A) Retry Outcomes by Error Code")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:,.0f}"))
ax2 = axes[1]
cats2=["Approved","Soft Failure\n(retriable)","Hard Failure\n(permanent)"]
vals2=[(df["failure_severity"]=="SUCCESS").sum()/n*100,
       (df["failure_severity"]=="SOFT_FAILURE").sum()/n*100,
       (df["failure_severity"]=="HARD_FAILURE").sum()/n*100]
bars2 = ax2.barh(cats2,vals2,color=[PALETTE["success"],PALETTE["soft"],PALETTE["hard"]],height=.5)
for b,v in zip(bars2,vals2):
    ax2.text(v+.3,b.get_y()+b.get_height()/2,f"{v:.1f}%",va="center",fontweight="bold",fontsize=12)
ax2.set_xlabel("% of All Transactions"); ax2.set_title("(B) Failure Decomposition")
plt.tight_layout(); fig.savefig("fig5_retry_segmentation.png",dpi=150,bbox_inches="tight")
print("  Saved: fig5_retry_segmentation.png"); plt.close()

# Fig 6 — Timeout temporal
top_banks = ["GTBank","First Bank","Zenith Bank","Access Bank"]
hb = (df[df["issuing_bank"].isin(top_banks)]
      .assign(to=(df["error_code"]=="96").astype(int))
      .groupby(["hour","issuing_bank"])["to"].mean().reset_index())
hb["to"]*=100
fig,ax = plt.subplots(figsize=(14,6))
colors6 = ["#E74C3C","#3498DB","#2ECC71","#9B59B6"]
for bank,col in zip(top_banks,colors6):
    sub = hb[hb["issuing_bank"]==bank]
    lw = 3. if bank=="GTBank" else 1.5
    ax.plot(sub["hour"],sub["to"],label=bank,color=col,lw=lw,
            marker="o" if bank=="GTBank" else None,ms=5)
ax.axvspan(19,21,alpha=.12,color="#E74C3C")
ax.text(20,hb["to"].max()*.88,"Peak\n19-21h",ha="center",fontsize=9,color="#E74C3C",fontweight="bold")
ax.set_xlabel("Hour of Day"); ax.set_ylabel("Timeout Error Rate (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("Error 96 (Timeout) Rate by Hour — GTBank Peak Spike\nRoot Cause: Slow 3DS Auth During Peak Load",
             fontweight="bold"); ax.legend(fontsize=10); ax.grid(True,alpha=.3)
plt.tight_layout(); fig.savefig("fig6_timeout_temporal.png",dpi=150,bbox_inches="tight")
print("  Saved: fig6_timeout_temporal.png"); plt.close()

# Fig 7 — Verve Grocery
grocery = df[df["merchant_category"]=="Grocery & Food"].copy()
grocery["period"] = np.where(grocery["day_of_sprint"]<45,"Before Day 45","After Day 45")
contingency = pd.crosstab(grocery["card_scheme"],grocery["is_failed"])
chi2,pval,dof,_ = stats.chi2_contingency(contingency)
print(f"\n  Chi-Square (Verve×Grocery): χ²={chi2:.1f}, p={pval:.2e}")
fig,axes = plt.subplots(1,2,figsize=(15,6))
fig.suptitle("Verve × Grocery MCC — Issuer Policy Block Analysis",fontsize=14,fontweight="bold")
ax = axes[0]
ps = grocery.groupby(["card_scheme","period"])["is_failed"].mean().mul(100).reset_index()
pivot_ps = ps.pivot(index="card_scheme",columns="period",values="is_failed")
pivot_ps.plot(kind="bar",ax=ax,color=["#3498DB","#E74C3C"],width=.6,edgecolor="white")
ax.set_xlabel("Card Scheme"); ax.set_ylabel("Failure Rate on Grocery MCC (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title("(A) Grocery Failure Rate: Before vs. After Day 45")
ax.set_xticklabels(ax.get_xticklabels(),rotation=0)
ax2 = axes[1]
daily = grocery[grocery["card_scheme"]=="Verve"].groupby("day_of_sprint")["is_failed"].mean().mul(100).reset_index()
ax2.plot(daily["day_of_sprint"],daily["is_failed"],color="#006B54",lw=2)
ax2.axvline(45,color=PALETTE["hard"],ls="--",lw=2,label="Issuer Policy Block (Day 45)")
ax2.fill_between(daily["day_of_sprint"],daily["is_failed"],alpha=.15,color="#006B54")
ax2.set_xlabel("Day of Sprint"); ax2.set_ylabel("Daily Failure Rate (%)")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_title("(B) Verve Daily Failure Rate on Grocery MCC\nAbrupt step-change on Day 45"); ax2.legend()
ax2.text(50,daily["is_failed"].max()*.7,f"p<0.001\n(χ²={chi2:.0f})",
         fontsize=10,color=PALETTE["hard"],fontweight="bold")
plt.tight_layout(); fig.savefig("fig7_verve_grocery.png",dpi=150,bbox_inches="tight")
print("  Saved: fig7_verve_grocery.png"); plt.close()

# Fig 8 — TPV Impact
top8 = pareto.nlargest(8,"tpv").copy(); top8["monthly_tpv_bn"]=top8["monthly_tpv"]/1e9
fig,ax = plt.subplots(figsize=(13,7))
bc8 = ["#2ECC71" if r else "#E74C3C" for r in top8["ret"]]
bars8 = ax.bar(top8["error_label"],top8["monthly_tpv_bn"],color=bc8,width=.6,edgecolor="white",lw=1.3)
for b in bars8:
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+.005,f"₦{b.get_height():.2f}B",
            ha="center",va="bottom",fontsize=10,fontweight="bold")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#2ECC71",label="Retriable"),
                   Patch(facecolor="#E74C3C",label="Non-Retriable")],fontsize=10)
ax.set_xlabel("Error Label"); ax.set_ylabel("Est. Monthly TPV at Risk (₦B)")
ax.set_title("Monthly TPV at Risk by Root-Cause Error\n₦960M/month recoverable across top 3 fixes",
             fontweight="bold")
plt.xticks(rotation=28,ha="right",fontsize=9)
plt.tight_layout(); fig.savefig("fig8_tpv_impact.png",dpi=150,bbox_inches="tight")
print("  Saved: fig8_tpv_impact.png"); plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — MODELLING
# ─────────────────────────────────────────────────────────────────────────────
print("\n[STEP 4] Training models...")
SAMPLE_N = 200_000
features = ["hour","is_weekend","is_peak_hour","scheme_code",
            "bank_first_bank","bank_gtbank","bank_zenith_bank",
            "bank_access_bank","bank_uba","verve_grocery_post45"]
sample = df[features+["is_failed"]].dropna().sample(SAMPLE_N,random_state=42)
X = sample[features]; y = sample["is_failed"]
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

# Logistic Regression
pipe = Pipeline([("sc",StandardScaler()),
                 ("clf",LogisticRegression(max_iter=500,random_state=42,class_weight="balanced"))])
pipe.fit(X_tr,y_tr)
y_pr = pipe.predict_proba(X_te)[:,1]
auc_lr = roc_auc_score(y_te,y_pr)
print(f"\n  Logistic Regression ROC-AUC: {auc_lr:.4f}")
print(classification_report(y_te,pipe.predict(X_te),target_names=["Success","Failure"]))

# Random Forest
rf = RandomForestClassifier(n_estimators=150,max_depth=7,random_state=42,n_jobs=-1,class_weight="balanced")
rf.fit(X_tr,y_tr)
y_pr_rf = rf.predict_proba(X_te)[:,1]
auc_rf = roc_auc_score(y_te,y_pr_rf)
print(f"  Random Forest ROC-AUC:       {auc_rf:.4f}")

# Fig 9 — ROC Curves + Feature Importance
fig,axes = plt.subplots(1,2,figsize=(15,6))
fig.suptitle("Model Performance & Feature Importance",fontsize=14,fontweight="bold")
ax = axes[0]
for y_p,label,col in [(y_pr,"Logistic Reg","#3498DB"),(y_pr_rf,"Random Forest","#E74C3C")]:
    fpr,tpr,_ = roc_curve(y_te,y_p); auc = roc_auc_score(y_te,y_p)
    ax.plot(fpr,tpr,color=col,lw=2.5,label=f"{label} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"--",color="grey",lw=1.5,label="Random")
ax.fill_between(*roc_curve(y_te,y_pr_rf)[:2],alpha=.08,color="#E74C3C")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("(A) ROC Curves — Failure Prediction Models"); ax.legend(fontsize=10)

ax2 = axes[1]
imp_df = pd.DataFrame({"feature":features,"imp":rf.feature_importances_}).sort_values("imp",ascending=True)
readable = {"hour":"Hour of Day","is_weekend":"Weekend","is_peak_hour":"Peak Hour (19-21h)",
            "scheme_code":"Card Scheme","bank_first_bank":"First Bank","bank_gtbank":"GTBank",
            "bank_zenith_bank":"Zenith Bank","bank_access_bank":"Access Bank","bank_uba":"UBA",
            "verve_grocery_post45":"Verve×Grocery×Post-45"}
imp_df["label"] = imp_df["feature"].map(readable)
colors9 = plt.cm.RdYlGn_r(np.linspace(.1,.9,len(imp_df)))
ax2.barh(imp_df["label"],imp_df["imp"],color=colors9,height=.6,edgecolor="white")
ax2.set_xlabel("Feature Importance (Gini)"); ax2.set_title("(B) Random Forest Feature Importance")
plt.tight_layout(); fig.savefig("fig9_model_performance.png",dpi=150,bbox_inches="tight")
print("  Saved: fig9_model_performance.png"); plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print(" ANALYSIS COMPLETE — KEY RESULTS")
print("="*65)
total_failures  = df["is_failed"].sum()
soft_failures   = (df["failure_severity"]=="SOFT_FAILURE").sum()
hard_failures   = (df["failure_severity"]=="HARD_FAILURE").sum()
hard_tpv        = df[df["failure_severity"]=="HARD_FAILURE"]["amount_ngn"].sum()
soft_tpv        = df[df["failure_severity"]=="SOFT_FAILURE"]["amount_ngn"].sum()
print(f" Raw TSR:              {1 - df['is_failed'].mean():.1%}")
print(f" Effective TSR:        {(df['effective_status']=='success').mean():.1%}")
print(f" Total Failures:       {total_failures:,} ({df['is_failed'].mean():.1%})")
print(f"   Soft (retriable):   {soft_failures:,} ({soft_failures/n:.1%})")
print(f"   Hard (permanent):   {hard_failures:,} ({hard_failures/n:.1%})")
print(f" Monthly Hard TPV Lost: ₦{hard_tpv/3:,.0f}")
print(f" Monthly Soft TPV Risk: ₦{soft_tpv/3:,.0f}")
print(f" LR ROC-AUC:   {auc_lr:.4f}")
print(f" RF ROC-AUC:   {auc_rf:.4f}")
print(f"\n TOP ROOT CAUSES:")
for _, row in pareto.head(3).iterrows():
    print(f"  Error {row['error_code']} ({row['error_label']}): {row['pct']:.1f}% of failures | Cumulative: {row['cum_pct']:.1f}%")
print("\n FILES GENERATED:")
figs = ["fig1_overview_dashboard.png","fig2_pareto_chart.png","fig3_issuer_scheme_heatmap.png",
        "fig4_mcc_scheme_heatmap.png","fig5_retry_segmentation.png","fig6_timeout_temporal.png",
        "fig7_verve_grocery.png","fig8_tpv_impact.png","fig9_model_performance.png"]
for f in figs: print(f"  {f}")
print("="*65)
print(" Run complete. All figures saved to working directory.")
print("="*65)


 PAYMENT GATEWAY — TRANSACTION SUCCESS RATE DIAGNOSIS
 Africa | Payments | 3-Week Sprint

[STEP 1] Generating synthetic transaction dataset...
  Sampling timestamps...
  Assigning attributes...
  Computing outcomes...

  Records: 2,400,000 | Raw TSR: 73.8% | Effective TSR: 77.0%

[STEP 2] Computing Pareto table...
error_code        error_label  count       pct    cum_pct   ret
        51 Insufficient Funds 195092 30.998327  30.998327  True
        96     System Timeout 157024 24.949671  55.947998  True
        05      Do Not Honour 145057 23.048225  78.996223 False
        14       Invalid Card  38084  6.051198  85.047421 False
        54       Expired Card  31491  5.003631  90.051052 False
        61      Exceeds Limit  18746  2.978567  93.029619 False
        91 Issuer Unavailable  18738  2.977296  96.006915  True
        57  Txn Not Permitted  12584  1.999482  98.006397 False
        41          Lost Card   6378  1.013406  99.019803 False
        43        Stolen Card   3117  0.4952

### Data Export and File Download
This section bundles the generated visualizations and exports the synthetic transaction database.

In [2]:
import zipfile
import os
from google.colab import files

# 1. Export the DataFrame to CSV
print("Exporting database to CSV (this may take a moment due to 2.4M rows)...")
df.to_csv('payment_transactions_data.csv', index=False)

# 2. Bundle all generated figures into a ZIP
figures = [
    "fig1_overview_dashboard.png", "fig2_pareto_chart.png", "fig3_issuer_scheme_heatmap.png",
    "fig4_mcc_scheme_heatmap.png", "fig5_retry_segmentation.png", "fig6_timeout_temporal.png",
    "fig7_verve_grocery.png", "fig8_tpv_impact.png", "fig9_model_performance.png"
]

with zipfile.ZipFile('payment_diagnosis_plots.zip', 'w') as zipf:
    for fig in figures:
        if os.path.exists(fig):
            zipf.write(fig)

print("Files prepared. Triggering downloads...")

# 3. Trigger downloads
files.download('payment_diagnosis_plots.zip')
files.download('payment_transactions_data.csv')

Exporting database to CSV (this may take a moment due to 2.4M rows)...
Files prepared. Triggering downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>